# PDF Q&A Assistant (LangChain RAG)

## 1. Project Overview

This notebook builds a **PDF Question-Answering assistant** using **LangChain** and a local retrieval pipeline.

It supports:
- Loading one or more PDFs
- Chunking long text into retrieval units
- Creating embeddings
- Storing/retrieving chunks
- Answering questions with cited context
- Comparing **naive prompting** vs **RAG**

Pipeline:

```
PDF files
  -> parse pages
  -> split into chunks
  -> embed chunks
  -> store in vector index
  -> retrieve top-k relevant chunks
  -> synthesize answer with citations
```


## 2. Learning Goals

By the end of this notebook, you will be able to:

1. Explain RAG (Retrieval-Augmented Generation) clearly
2. Load and parse multiple PDF documents
3. Chunk documents for retrieval quality
4. Build embeddings and a chunk index
5. Retrieve relevant chunks for a question
6. Generate cited answers from retrieved evidence
7. Compare naive prompting against RAG behavior

## 3. Problem Statement

LLMs often hallucinate when asked questions without reliable context.

Example:
- Question: *"What does the PDF say about transformer scaling laws?"*
- If the model does not have the PDF content in context, it may answer from generic memory.

We need a method that grounds answers in the actual documents. RAG solves this by retrieving relevant chunks and forcing the answer to use those chunks as evidence.

## 4. Why This Project Matters

RAG is the default architecture for enterprise GenAI assistants because it:

- Reduces hallucination by grounding answers in source docs
- Enables citation-based responses for trust and auditability
- Works with proprietary documents without retraining foundation models
- Supports continuously updated knowledge by re-indexing docs

Practical use cases:
- Policy assistants
- Contract review copilots
- Research PDF assistants
- Internal knowledge bots

## 5. Dataset Overview

We use two public PDFs downloaded directly in this notebook:

1. **Attention Is All You Need** (arXiv, 2017)
2. **BERT: Pre-training of Deep Bidirectional Transformers** (arXiv, 2018)

Source links:
- https://arxiv.org/pdf/1706.03762.pdf
- https://arxiv.org/pdf/1810.04805.pdf

Both are public research papers and suitable for RAG demos.

## 6. Environment / Setup

This notebook uses LangChain components with local embeddings. No API key is required.

Optional external service:
- **Ollama** (optional) for improved natural-language answer synthesis
- If not available, the notebook falls back to deterministic citation-based synthesis

In [1]:
import os
import json
import math
import urllib.request
from pathlib import Path
from typing import List, Dict, Any, Tuple

import numpy as np

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Optional dependencies used if available
try:
    from langchain_community.embeddings import HuggingFaceEmbeddings
    HF_EMBEDDINGS_AVAILABLE = True
except Exception:
    HF_EMBEDDINGS_AVAILABLE = False

try:
    from langchain_chroma import Chroma
    CHROMA_NEW_API = True
except Exception:
    CHROMA_NEW_API = False

if not CHROMA_NEW_API:
    from langchain_community.vectorstores import Chroma


# Optional LLM client (Ollama)
try:
    from langchain_ollama import ChatOllama
    OLLAMA_LIB_AVAILABLE = True
except Exception:
    OLLAMA_LIB_AVAILABLE = False

print("Environment ready")
print(f"HF embeddings available: {HF_EMBEDDINGS_AVAILABLE}")
print(f"Ollama library available: {OLLAMA_LIB_AVAILABLE}")


C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Environment ready
HF embeddings available: True
Ollama library available: True


## 7. Imports

Core libraries already imported in setup cell:
- LangChain loaders/splitters/vector stores
- NumPy for fallback vector math
- urllib/pathlib for PDF download and paths


In [2]:
print("LangChain PDF RAG stack imported successfully.")

LangChain PDF RAG stack imported successfully.


## 8. Data Loading — Download PDFs in Notebook

This cell downloads PDFs directly so the notebook is reproducible end-to-end.

In [3]:
ROOT = Path.cwd()
DATA_DIR = ROOT / "pdf_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

PDF_SOURCES = [
    {
        "name": "attention_is_all_you_need",
        "url": "https://arxiv.org/pdf/1706.03762.pdf",
        "file": DATA_DIR / "attention_is_all_you_need.pdf",
    },
    {
        "name": "bert_pretraining",
        "url": "https://arxiv.org/pdf/1810.04805.pdf",
        "file": DATA_DIR / "bert_pretraining.pdf",
    },
]

for src in PDF_SOURCES:
    if src["file"].exists() and src["file"].stat().st_size > 0:
        print(f"Exists: {src['file'].name} ({src['file'].stat().st_size:,} bytes)")
    else:
        print(f"Downloading: {src['url']}")
        urllib.request.urlretrieve(src["url"], src["file"])
        print(f"Saved: {src['file'].name} ({src['file'].stat().st_size:,} bytes)")

PDF_PATHS = [str(src["file"]) for src in PDF_SOURCES]
print("\nPDF files:")
for p in PDF_PATHS:
    print(" -", p)


Exists: attention_is_all_you_need.pdf (2,215,244 bytes)
Exists: bert_pretraining.pdf (775,166 bytes)

PDF files:
 - E:\Github\Machine-Learning-Projects\GenAI\01_PDF_QA_Assistant\pdf_data\attention_is_all_you_need.pdf
 - E:\Github\Machine-Learning-Projects\GenAI\01_PDF_QA_Assistant\pdf_data\bert_pretraining.pdf


## 9. Data Inspection / EDA

We load all pages and inspect volume, page counts, and sample text.

In [4]:
raw_docs = []
for pdf_path in PDF_PATHS:
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    for d in docs:
        d.metadata["source_file"] = Path(pdf_path).name
    raw_docs.extend(docs)

print(f"Total pages loaded: {len(raw_docs)}")

# Page count per source
page_count = {}
for d in raw_docs:
    src = d.metadata.get("source_file", "unknown")
    page_count[src] = page_count.get(src, 0) + 1

print("Pages per PDF:")
for k, v in page_count.items():
    print(f"  {k}: {v}")

# Show snippets
print("\nSample snippets:")
for i in [0, min(5, len(raw_docs)-1), min(15, len(raw_docs)-1)]:
    text = raw_docs[i].page_content.replace("\n", " ")
    print(f"\nDoc[{i}] source={raw_docs[i].metadata.get('source_file')} page={raw_docs[i].metadata.get('page')}")
    print(text[:300] + "...")


Total pages loaded: 31
Pages per PDF:
  attention_is_all_you_need.pdf: 15
  bert_pretraining.pdf: 16

Sample snippets:

Doc[0] source=attention_is_all_you_need.pdf page=0
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Par...

Doc[5] source=attention_is_all_you_need.pdf page=5
Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations for different layer types. n is the sequence length, d is the representation dimension, k is the kernel size of convolutions and r the size of the neighborhood in restricted self-attention. Layer Type Com...

Doc[15] source=bert_pretraining.pdf page=0
BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding Jacob Devlin Ming-Wei Chang Kenton Lee Krist

## 10. Data Preprocessing — Chunking

RAG works on chunks, not whole documents.

Why chunking matters:
- Too large: retrieval is noisy, context windows wasted
- Too small: information is fragmented and loses meaning

We start with:
- `chunk_size=900`
- `chunk_overlap=150`


In [5]:
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunks = splitter.split_documents(raw_docs)

for idx, ch in enumerate(chunks):
    ch.metadata["chunk_id"] = idx

print(f"Chunks created: {len(chunks)}")
lengths = [len(c.page_content) for c in chunks]
print(f"Chunk length stats -> min={min(lengths)}, mean={sum(lengths)/len(lengths):.1f}, max={max(lengths)}")

print("\nFirst chunk metadata:")
print(chunks[0].metadata)
print("\nFirst chunk preview:")
print(chunks[0].page_content[:350] + "...")


Chunks created: 146
Chunk length stats -> min=170, mean=806.6, max=899

First chunk metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'E:\\Github\\Machine-Learning-Projects\\GenAI\\01_PDF_QA_Assistant\\pdf_data\\attention_is_all_you_need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention_is_all_you_need.pdf', 'chunk_id': 0}

First chunk preview:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Re

## 11. Baseline Approach (Naive Prompting)

Naive approach = answer question directly from a general model (or from no retrieval context).

In this notebook, naive baseline is simulated as:
- No retrieval step
- Simple direct response template / optional LLM direct answer

This gives us a clear comparison point against RAG.

In [6]:
def naive_answer(question: str) -> str:
    """Naive baseline without retrieval grounding."""
    return (
        "[Naive baseline] This answer is not grounded in retrieved PDF context. "
        f"Question was: '{question}'. "
        "Without retrieval, responses may be incomplete or hallucinated."
    )

q_demo = "What are the key advantages of self-attention over recurrence?"
print("Question:", q_demo)
print("Naive answer:")
print(naive_answer(q_demo))


Question: What are the key advantages of self-attention over recurrence?
Naive answer:
[Naive baseline] This answer is not grounded in retrieved PDF context. Question was: 'What are the key advantages of self-attention over recurrence?'. Without retrieval, responses may be incomplete or hallucinated.


## 12. Main Workflow — Embeddings + Vector Store

We build embeddings and store chunk vectors for semantic retrieval.

Primary path:
- `HuggingFaceEmbeddings` (`sentence-transformers/all-MiniLM-L6-v2`)
- Chroma vector store

Fallback path (if embedding model download fails):
- Simple TF-IDF vectors (deterministic)
- In-memory cosine retrieval


In [7]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RAG_BACKEND = None

# Objects used across the notebook
vectorstore = None
retriever = None
fallback_chunk_matrix = None
fallback_vocab = None


def _tfidf_build(texts: List[str]) -> Tuple[np.ndarray, Dict[str, int], np.ndarray]:
    tokenized = []
    df = {}
    for t in texts:
        tokens = [w.lower() for w in t.split() if w.isalpha() or w.replace("-", "").isalpha()]
        tokenized.append(tokens)
        seen = set(tokens)
        for tok in seen:
            df[tok] = df.get(tok, 0) + 1

    vocab = {tok: i for i, tok in enumerate(df.keys())}
    n_docs = len(texts)
    idf = np.zeros(len(vocab), dtype=np.float32)
    for tok, i in vocab.items():
        idf[i] = math.log((1 + n_docs) / (1 + df[tok])) + 1.0

    mat = np.zeros((n_docs, len(vocab)), dtype=np.float32)
    for r, toks in enumerate(tokenized):
        if not toks:
            continue
        tf = {}
        for tok in toks:
            if tok in vocab:
                tf[tok] = tf.get(tok, 0) + 1
        for tok, cnt in tf.items():
            c = vocab[tok]
            mat[r, c] = (cnt / len(toks)) * idf[c]

    # L2 normalize rows
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-9
    mat = mat / norms
    return mat, vocab, idf


def _tfidf_query_vec(question: str, vocab: Dict[str, int], idf: np.ndarray) -> np.ndarray:
    toks = [w.lower() for w in question.split() if w.isalpha() or w.replace("-", "").isalpha()]
    vec = np.zeros(len(vocab), dtype=np.float32)
    if not toks:
        return vec
    tf = {}
    for tok in toks:
        if tok in vocab:
            tf[tok] = tf.get(tok, 0) + 1
    for tok, cnt in tf.items():
        c = vocab[tok]
        vec[c] = (cnt / len(toks)) * idf[c]
    vec /= (np.linalg.norm(vec) + 1e-9)
    return vec


if HF_EMBEDDINGS_AVAILABLE:
    try:
        embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory=str(ROOT / "chroma_pdf_qa"),
        )
        retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
        RAG_BACKEND = "langchain_chroma_hf"
        print("Using LangChain + Chroma + HuggingFace embeddings")
    except Exception as e:
        print(f"HF/Chroma path unavailable, switching to TF-IDF fallback. Reason: {e}")

if RAG_BACKEND is None:
    texts = [c.page_content for c in chunks]
    fallback_chunk_matrix, fallback_vocab, fallback_idf = _tfidf_build(texts)
    RAG_BACKEND = "tfidf_fallback"
    print("Using fallback backend: TF-IDF in-memory retrieval")

print("RAG backend:", RAG_BACKEND)


C:\Users\ahmad\AppData\Local\Temp\ipykernel_41100\2600513343.py:63: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using LangChain + Chroma + HuggingFace embeddings
RAG backend: langchain_chroma_hf


## 13. Training / Inference / Pipeline Execution

For RAG, execution means indexing + retrieval + answer synthesis (no model fine-tuning required).

This cell defines retrieval and citation-aware answer functions.

In [8]:
def retrieve_chunks(question: str, k: int = 4):
    if RAG_BACKEND == "langchain_chroma_hf":
        docs = retriever.invoke(question)
        return docs[:k]

    # TF-IDF fallback retrieval
    qv = _tfidf_query_vec(question, fallback_vocab, fallback_idf)
    sims = fallback_chunk_matrix @ qv
    top_idx = np.argsort(-sims)[:k]
    docs = []
    for idx in top_idx:
        d = chunks[int(idx)]
        d.metadata["similarity"] = float(sims[idx])
        docs.append(d)
    return docs


def format_citations(docs: List[Any]) -> List[str]:
    cits = []
    for d in docs:
        src = d.metadata.get("source_file", "unknown")
        page = d.metadata.get("page", "?")
        chunk_id = d.metadata.get("chunk_id", "?")
        cits.append(f"{src} | page {page} | chunk {chunk_id}")
    return cits


def synthesize_answer(question: str, docs: List[Any]) -> str:
    # Deterministic extractive synthesis from retrieved chunks
    snippets = []
    for d in docs:
        txt = " ".join(d.page_content.split())
        snippets.append(txt[:350])

    if not snippets:
        return "I could not retrieve relevant context."

    answer = (
        "Based on retrieved PDF context, here is a grounded answer:\n\n"
        f"Question: {question}\n\n"
        f"Evidence summary:\n- {snippets[0]}"
    )
    if len(snippets) > 1:
        answer += f"\n- {snippets[1]}"
    return answer


def rag_answer(question: str, k: int = 4) -> Dict[str, Any]:
    docs = retrieve_chunks(question, k=k)
    answer = synthesize_answer(question, docs)
    citations = format_citations(docs)
    return {
        "question": question,
        "answer": answer,
        "citations": citations,
        "docs": docs,
    }


q = "What are the key advantages of self-attention over recurrent models?"
out = rag_answer(q, k=4)
print("Question:", out["question"])
print("\nAnswer:\n", out["answer"])
print("\nCitations:")
for c in out["citations"]:
    print(" -", c)


Question: What are the key advantages of self-attention over recurrent models?

Answer:
 Based on retrieved PDF context, here is a grounded answer:

Question: What are the key advantages of self-attention over recurrent models?

Evidence summary:
- self-attention and discuss its advantages over models such as [17, 18] and [9]. 3 Model Architecture Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 35]. Here, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a sequence of continuous representations z = ( z1, ..., zn). Given
- self-attention and discuss its advantages over models such as [17, 18] and [9]. 3 Model Architecture Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 35]. Here, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a sequence of continuous representations z = ( z1, ..., zn). Given

Citations:
 - attention_is_all_you_need.pdf | 

## 14. Evaluation and Interpretation

We compare naive prompting vs RAG on the same question set.

Interpretation criteria:
- **Grounding:** does the answer reference retrieved evidence?
- **Specificity:** does it mention PDF-specific details?
- **Citations:** can we trace claims to source chunks?

In [9]:
EVAL_QUESTIONS = [
    "Why does the Transformer reduce sequential computation compared to RNNs?",
    "What pre-training objectives are used in BERT?",
    "How do these papers discuss attention complexity?",
]

results = []
for q in EVAL_QUESTIONS:
    naive = naive_answer(q)
    rag = rag_answer(q, k=4)
    results.append({"question": q, "naive": naive, "rag": rag})

for i, r in enumerate(results, 1):
    print("=" * 100)
    print(f"Q{i}: {r['question']}")
    print("\n[Naive]\n" + r["naive"])
    print("\n[RAG]\n" + r["rag"]["answer"])
    print("\n[RAG citations]")
    for c in r["rag"]["citations"]:
        print(" -", c)

print("\nInterpretation:")
print("RAG answers are grounded by retrieved chunks and include traceable citations.")


Q1: Why does the Transformer reduce sequential computation compared to RNNs?

[Naive]
[Naive baseline] This answer is not grounded in retrieved PDF context. Question was: 'Why does the Transformer reduce sequential computation compared to RNNs?'. Without retrieval, responses may be incomplete or hallucinated.

[RAG]
Based on retrieved PDF context, here is a grounded answer:

Question: Why does the Transformer reduce sequential computation compared to RNNs?

Evidence summary:
- are used in conjunction with a recurrent network. In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in transl
- are used in conjunction with a recurrent network. In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on 

## 15. Error Analysis / Limitations

Common failure modes in PDF RAG:

1. **Bad chunk boundaries**
   - Important sentence split across chunks
   - Fix: adjust chunk overlap

2. **Weak embedding model for technical terms**
   - Retrieval misses relevant paragraphs
   - Fix: try domain-specific embedding models

3. **OCR / parsing artifacts in PDFs**
   - Broken text harms embeddings and retrieval
   - Fix: OCR cleanup and filtering

4. **Citation granularity**
   - Chunk-level citation may be too coarse
   - Fix: sentence-level post-processing and span highlighting

5. **No true generative LLM in fallback mode**
   - Deterministic synthesis is robust but less fluent
   - Fix: plug in local Ollama model for richer final answers


## 16. How to Improve It

### Production Improvement Ideas

1. Add query rewriting (HyDE / multi-query retrieval)
2. Use rerankers after initial retrieval (cross-encoder)
3. Add metadata filtering (source, section, year)
4. Add conversation memory for multi-turn Q&A
5. Cache embeddings and retrieval results
6. Add evaluation set with answer faithfulness metrics
7. Add UI (Streamlit/Gradio) with source-highlighting


## Common Mistakes

1. **Using huge chunk size blindly**
   - Retrieval becomes noisy and misses focused evidence.

2. **Ignoring overlap**
   - Important context gets split and lost.

3. **No citations in final answer**
   - Users cannot verify claims.

4. **Comparing RAG to no baseline**
   - Hard to show the value of retrieval.

5. **Trusting top-1 retrieval only**
   - Use top-k and synthesize across evidence.


## 17. Practice Exercises

### Mini Challenge

1. Change `CHUNK_SIZE` from 900 to 400 and 1400. Compare answer quality.
2. Swap embedding model from `all-MiniLM-L6-v2` to another sentence-transformers model.
3. Change retriever `k` from 4 to 2 and 8. Observe citation relevance.
4. Add a third PDF and test cross-document questions.

### Try This Next

- Add a reranker step and compare before/after retrieval quality.
- Add confidence scoring from retrieval similarities.
- Build a simple chat loop with preserved conversation state.


## 18. Final Takeaway / Summary

You built a complete PDF RAG assistant with LangChain:

- Loaded and parsed multiple PDFs
- Chunked document pages for retrieval
- Built embeddings and an index
- Retrieved relevant chunks per question
- Produced citation-grounded answers
- Compared naive prompting vs RAG

Core insight:

> Naive prompting answers from model prior knowledge; RAG answers from retrievable evidence.

That grounding step is what makes production Q&A assistants trustworthy.